## Gringotts Complaint Triage — LLM-Orchestrated Medallion Pipeline

A CFPB consumer complaint dataset run through a full Bronze → Silver → Gold
pipeline, using an LLM-as-judge pattern to generate and evaluate simulated
complaint responses at scale.

The Hogwarts/Gringotts framing is cosmetic — the "Houses" are really four
response personas (advocacy, analysis, practical resolution, strategy) and
"Dumbledore" is an LLM-as-judge evaluator. Underneath the theme, this
notebook is a demonstration of a few things that matter more in production
LLM pipelines than the prompts themselves:

**1. Deterministic logic and LLM inference are kept strictly separate.**
Routing (which House a complaint goes to) is a rules-based lookup on the
CFPB `Issue` field, not an LLM decision — the model is told the routing is
final and not to reinterpret it. Scoring is the same pattern one layer up:
Dumbledore returns raw per-dimension scores, but the weighted composite and
final House Cup points are computed in Spark. Anything that needs to be
reproducible and auditable stays out of the LLM's hands; anything that
genuinely requires language understanding (sentiment, summarization,
persona-driven writing, qualitative evaluation) goes to Gemini.

**2. Generation and evaluation are separate calls.** The House agent that
writes a response and "Dumbledore" who grades it are two independent LLM
calls — a model doesn't score the response it just wrote in the same
context, which avoids the self-grading bias that tends to inflate LLM-judge
scores.

**3. Every LLM call is fully observable.** Every enrichment step returns a
typed error code, HTTP status, retry count, model version, and latency
alongside the actual output, win or fail. A batch run is debuggable from the
resulting table without re-hitting the API.

**4. Failure handling is layered.** Transient errors (timeouts, 429/5xx)
retry with exponential backoff; malformed schema, bad JSON, or a token-limit
hit fail fast instead of burning retries on an error that won't resolve
itself.

### Pipeline

| Layer | What happens |
|---|---|
| **Bronze** | Pull a sample of complaints with narrative text from the Databricks Marketplace zero-ETL share |
| **Silver** | Deterministic House routing (rules-based) + Gemini sentiment/summary enrichment |
| **Stage 2** | Per-House persona response generation → independent Dumbledore evaluation → Spark-computed quality scoring |
| **Gold** | Business-friendly renamed views, retaining full operational metadata |
| **Dashboard Views** | Presentation-only projections and rollups — no inference happens here |

Small `LIMIT` values throughout keep this cheap and fast to re-run as a demo;
none of the sizing choices here are meant as production guidance.

### Bronze Layer — Environment Setup

Creating a dedicated `gringotts` schema keeps this portfolio project isolated from
other workspace assets — all tables, views, and checkpoints for this pipeline
live here and can be dropped cleanly without touching anything else.

Databricks Marketplace exposes the CFPB Consumer Complaint Database as a
live share, so there's no extract/load step to write — the data is queried
directly from the shared catalog.

This step pulls a small sample (20 rows) and filters out complaints with no
narrative text, since the narrative field is what downstream NLP/LLM steps
in this notebook will operate on. The row limit keeps the demo fast and
cheap to re-run; it's not a production ingestion pattern.

Writing the sampled data to a managed Delta table gives this pipeline a
stable, versioned Bronze layer — later steps read from `gringotts.bronze_complaints`
rather than re-querying the Marketplace share, which decouples the rest of
the notebook from the source's availability and schema drift.

`overwriteSchema` is set so repeated runs during development don't fail on
schema mismatches.

Quick confirmation that the narrative text made it into Bronze intact
before moving on to the Silver layer.

In [0]:
# Gringotts Bronze Layer

# 1. Ensure isolated database/schema exists for portfolio cleanliness
spark.sql("CREATE SCHEMA IF NOT EXISTS gringotts")
spark.sql("USE gringotts")

# 2. Read directly from the Zero-ETL Databricks Marketplace share
bronze_df = spark.sql("""
    SELECT * 
    FROM consumer_complaints.consumer_complaints.consumer_complaint_database
    WHERE Consumer_Complaint_Narrative IS NOT NULL 
      AND Consumer_Complaint_Narrative != ''
    LIMIT 20
""")

# 3. Save to your local portfolio Bronze table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gringotts.bronze_complaints")

# Verify the text is there
display(spark.sql("SELECT Complaint_ID, Consumer_Complaint_Narrative FROM gringotts.bronze_complaints"))

### Silver Layer — Routing & AI Enrichment

This step does two independent things to every Bronze complaint, and keeps them
deliberately separate:

**1. Deterministic house routing.** Each complaint's CFPB `Issue` field is mapped
to one of four "houses" via a fixed lookup table (fraud → Slytherin, servicing/fees
→ Hufflepuff, etc.). Issues with no explicit mapping are assigned to whichever
house currently has the fewest complaints, so coverage stays balanced rather than
piling into one bucket. This routing is rule-based on purpose — an LLM never
touches it, keeping the assignment auditable and reproducible.

**2. LLM-based enrichment.** Each complaint narrative is sent to Gemini 3.8 Flash
to produce a sentiment label (`Positive` → `Severe`) and a short wizarding-style
summary. The prompt explicitly tells Gemini the house has already been decided
and not to reinterpret it — the model's job is narrowly scoped to sentiment and
summarization, nothing else.

The Gemini call is wrapped in a retrying UDF: transient failures (timeouts,
connection errors, HTTP 429/500/502/503/504) get exponential backoff up to
`MAX_ATTEMPTS`, while non-retryable failures (schema violations, malformed JSON,
hitting the token limit) fail fast with a typed error code. Every row's status,
error detail, HTTP code, retry count, and latency are captured alongside the AI
output, so a batch run is fully diagnosable after the fact rather than surfacing
as a bare exception. `repartition(1)` intentionally serializes the calls — a
demo-scale throttle to respect Gemini's rate limits, not a production pattern.

The result is flattened and written to `gringotts.silver_triage`, then displayed
for a quick check that routing, sentiment, and summary all landed as expected.

In [0]:
# Gringotts Silver layer
# Processes and enriches data

import json
import random
import time
import requests
from collections import Counter

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import udf

# Configuration
MODEL = "gemini-3.8-flash"
API_KEY = dbutils.secrets.get(catalog="workspace", schema="gringotts", key="gemini_api_key")
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
MAX_ATTEMPTS = 4
REQUEST_TIMEOUT = 60
MAX_NARRATIVE_CHARS = 600
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}
HOUSE_ORDER = ["Gryffindor", "Ravenclaw", "Hufflepuff", "Slytherin"]

# CFPB Issue -> Hogwarts House mapping. Issue is the routing signal; Gemini never assigns house.
ISSUE_TO_HOUSE = {
    # Gryffindor — disputes, escalation, collection conflict
    "Collection practices": "Gryffindor",
    "Collection debt dispute": "Gryffindor",
    "Communication tactics": "Gryffindor",
    "Cant contact lender": "Gryffindor",
    "Takingthreatening an illegal action": "Gryffindor",
    "Improper contact or sharing of info": "Gryffindor",
    "Contd attempts collect debt not owed": "Gryffindor",
    "Arbitration": "Gryffindor",
    # Ravenclaw — reporting, rates, disclosures, credit decisions
    "Billing statement": "Ravenclaw",
    "Billing disputes": "Ravenclaw",
    "Credit reporting companys investigation": "Ravenclaw",
    "Credit reporting": "Ravenclaw",
    "Incorrect information on credit report": "Ravenclaw",
    "Unable to get credit reportcredit score": "Ravenclaw",
    "Improper use of my credit report": "Ravenclaw",
    "APR or interest rate": "Ravenclaw",
    "Credit decision  Underwriting": "Ravenclaw",
    "Credit determination": "Ravenclaw",
    "Incorrectmissing disclosures or info": "Ravenclaw",
    "Disclosures": "Ravenclaw",
    "Disclosure verification of debt": "Ravenclaw",
    "Account terms and changes": "Ravenclaw",
    "Incorrect exchange rate": "Ravenclaw",
    # Hufflepuff — servicing, fees, hardship
    "Fees": "Hufflepuff",
    "Other fee": "Hufflepuff",
    "UnexpectedOther fees": "Hufflepuff",
    "Excessive fees": "Hufflepuff",
    "Late fee": "Hufflepuff",
    "Overlimit fee": "Hufflepuff",
    "Balance transfer fee": "Hufflepuff",
    "Cash advance fee": "Hufflepuff",
    "Charged fees or interest I didnt expect": "Hufflepuff",
    "Customer serviceCustomer relations": "Hufflepuff",
    "Customer service  Customer relations": "Hufflepuff",
    "Other service issues": "Hufflepuff",
    "Managing the loan or lease": "Hufflepuff",
    "Managing the line of credit": "Hufflepuff",
    "Loan servicing payments escrow account": "Hufflepuff",
    "Dealing with my lender or servicer": "Hufflepuff",
    "Problems when you are unable to pay": "Hufflepuff",
    "Cant repay my loan": "Hufflepuff",
    "Forbearance  Workout plans": "Hufflepuff",
    "Repaying your loan": "Hufflepuff",
    # Slytherin — fraud, identity theft, unauthorized activity
    "Identity theft  Fraud  Embezzlement": "Slytherin",
    "Fraud or scam": "Slytherin",
    "Received a loan I didnt apply for": "Slytherin",
    "Unauthorized transactionstrans. issues": "Slytherin",
    "Lost or stolen check": "Slytherin",
    "Lost or stolen money order": "Slytherin",
    "Lost or stolen money": "Slytherin",
    "Privacy": "Slytherin",
    "Credit card protection  Debt protection": "Slytherin",
    "Credit monitoring or identity protection": "Slytherin",
}

# Read Bronze
bronze_df = (
    spark.table("gringotts.bronze_complaints")
    .filter(F.col("Consumer_Complaint_Narrative").isNotNull())
    .filter(F.trim(F.col("Consumer_Complaint_Narrative")) != "")
)

# Deterministic routing: known issues map directly; unmapped go to the least-populated house
issue_rows = (
    bronze_df
    .select(F.col("Complaint_ID").cast("string").alias("Complaint_ID"), F.col("Issue"))
    .collect()
)

house_counts = Counter({h: 0 for h in HOUSE_ORDER})
for row in issue_rows:
    if row["Issue"] in ISSUE_TO_HOUSE:
        house_counts[ISSUE_TO_HOUSE[row["Issue"]]] += 1
print("Initial deterministic house distribution:", dict(house_counts))

routing_rows = []
current_counts = house_counts.copy()
for row in issue_rows:
    issue = row["Issue"]
    if issue in ISSUE_TO_HOUSE:
        routed_house, routing_method = ISSUE_TO_HOUSE[issue], "DETERMINISTIC_ISSUE"
    else:
        routed_house = min(HOUSE_ORDER, key=lambda h: (current_counts[h], HOUSE_ORDER.index(h)))
        routing_method = "BALANCED_UNMAPPED_ISSUE"
        current_counts[routed_house] += 1
    routing_rows.append((row["Complaint_ID"], routed_house, routing_method))

routing_df = spark.createDataFrame(
    routing_rows,
    schema=StructType([
        StructField("Complaint_ID", StringType(), True),
        StructField("routed_house", StringType(), True),
        StructField("routing_method", StringType(), True),
    ]),
)

print("\nFinal routing distribution:")
routing_df.groupBy("routing_method", "routed_house").count().orderBy("routing_method", "routed_house").show()

ai_input_df = (
    bronze_df
    .select(F.col("Complaint_ID").cast("string").alias("Complaint_ID"), "Issue", "Consumer_Complaint_Narrative")
    .join(routing_df, on="Complaint_ID", how="inner")
)

# Gemini contract. House is NOT part of the response — routing is already decided.
response_schema = {
    "type": "OBJECT",
    "properties": {
        "sentiment": {"type": "STRING", "enum": ["Mild", "Negative", "Severe", "Neutral", "Positive"]},
        "gringotts_summary": {"type": "STRING"},
    },
    "required": ["sentiment", "gringotts_summary"],
}

triage_schema = StructType([
    StructField("triage_status", StringType(), True),
    StructField("sentiment", StringType(), True),
    StructField("gringotts_summary", StringType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("retry_count", IntegerType(), True),
    StructField("model_version", StringType(), True),
    StructField("elapsed_seconds", DoubleType(), True),
])


def analyze_complaint(complaint_id, issue, narrative, routed_house, routing_method):
    start_time = time.time()
    result = {
        "triage_status": "ERROR", "sentiment": None, "gringotts_summary": None,
        "error_type": None, "error_message": None, "http_status": None,
        "retry_count": 0, "model_version": MODEL, "elapsed_seconds": 0.0,
    }

    def _return():
        result["elapsed_seconds"] = time.time() - start_time
        return tuple(result[f.name] for f in triage_schema.fields)

    if narrative is None or not str(narrative).strip():
        result["error_type"] = "EMPTY_NARRATIVE"
        result["error_message"] = "Complaint narrative is empty."
        return _return()

    narrative = str(narrative)[:MAX_NARRATIVE_CHARS]
    prompt = f"""
You are the Goblin Triage analyst for a fictional wizarding bank.
Analyze the following consumer financial complaint.
The CFPB has already categorized it with:
Issue: {issue}
It has already been routed to:
House: {routed_house}
Routing method: {routing_method}
Do NOT change or reinterpret the house assignment.
Determine only:
1. Sentiment — Positive / Neutral / Mild / Negative / Severe (escalating scale)
2. A concise Gringotts-style summary, 15 words or fewer, no invented facts

Complaint narrative:
{narrative}
"""
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "thinkingConfig": {"thinkingLevel": "low"},
            "maxOutputTokens": 1024,
            "responseMimeType": "application/json",
            "responseSchema": response_schema,
        },
    }
    headers = {"Content-Type": "application/json", "x-goog-api-key": API_KEY}

    def _retry_delay(attempt, response=None):
        retry_after = response.headers.get("Retry-After") if response is not None else None
        if retry_after:
            try:
                return float(retry_after)
            except ValueError:
                pass
        return 2 ** (attempt - 1) + random.random()

    for attempt in range(1, MAX_ATTEMPTS + 1):
        result["retry_count"] = attempt - 1
        try:
            response = requests.post(ENDPOINT, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            result["http_status"] = response.status_code

            if response.status_code == 200:
                try:
                    response_json = response.json()
                except Exception as e:
                    result["error_type"], result["error_message"] = "RESPONSE_PARSE_ERROR", str(e)
                    if attempt < MAX_ATTEMPTS:
                        time.sleep(_retry_delay(attempt))
                        continue
                    break

                result["model_version"] = response_json.get("modelVersion", MODEL)
                candidates = response_json.get("candidates", [])
                if not candidates:
                    result["error_type"], result["error_message"] = "NO_CANDIDATE", "Gemini returned no candidates."
                    break

                candidate = candidates[0]
                if candidate.get("finishReason") == "MAX_TOKENS":
                    result["error_type"] = "MAX_OUTPUT_TOKENS"
                    result["error_message"] = "Gemini exhausted the output token budget."
                    break

                text_parts = [p["text"] for p in candidate.get("content", {}).get("parts", []) if "text" in p]
                response_text = "".join(text_parts).strip()
                if not response_text:
                    result["error_type"], result["error_message"] = "EMPTY_RESPONSE", "Gemini returned no visible response text."
                    break

                try:
                    parsed = json.loads(response_text)
                except Exception as e:
                    result["error_type"], result["error_message"] = "MODEL_JSON_PARSE_ERROR", str(e)
                    break

                sentiment, summary = parsed.get("sentiment"), parsed.get("gringotts_summary")
                valid_sentiments = {"Mild", "Negative", "Severe", "Neutral", "Positive"}
                if sentiment not in valid_sentiments:
                    result["error_type"], result["error_message"] = "SCHEMA_ERROR", f"Invalid sentiment returned: {sentiment}"
                    break
                if not isinstance(summary, str) or not summary.strip():
                    result["error_type"], result["error_message"] = "SCHEMA_ERROR", "Missing or empty gringotts_summary."
                    break

                result.update(triage_status="SUCCESS", sentiment=sentiment,
                              gringotts_summary=summary.strip(), error_type=None, error_message=None)
                return _return()

            # Non-200 HTTP response
            result["error_type"] = f"HTTP_{response.status_code}"
            result["error_message"] = response.text[:2000]
            if response.status_code in RETRYABLE_STATUS_CODES and attempt < MAX_ATTEMPTS:
                time.sleep(_retry_delay(attempt, response))
                continue
            break

        except requests.exceptions.Timeout as e:
            result["error_type"], result["error_message"] = "TIMEOUT", str(e)
        except requests.exceptions.RequestException as e:
            result["error_type"], result["error_message"] = "CONNECTION_ERROR", str(e)
        except Exception as e:
            result["error_type"], result["error_message"] = "UNEXPECTED_ERROR", str(e)
            break

        if attempt < MAX_ATTEMPTS:
            time.sleep(_retry_delay(attempt))

    return _return()


analyze_complaint_udf = udf(analyze_complaint, triage_schema)

# repartition(1) is a demo-scale concurrency cap, not a production rate limiter
silver_df = (
    ai_input_df
    .repartition(1)
    .withColumn("triage", analyze_complaint_udf(
        F.col("Complaint_ID"), F.col("Issue"), F.col("Consumer_Complaint_Narrative"),
        F.col("routed_house"), F.col("routing_method"),
    ))
    .select("Complaint_ID", "Issue", "Consumer_Complaint_Narrative",
            "routed_house", "routing_method", "triage.*")
)

(silver_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gringotts.silver_triage"))

display(spark.sql("""
    SELECT Complaint_ID, Issue, routed_house, routing_method,
           sentiment, gringotts_summary, triage_status,
           error_type, retry_count, model_version, elapsed_seconds
    FROM gringotts.silver_triage
    ORDER BY Complaint_ID
"""))

### Gold Layer — Business-Ready Presentation

Silver already holds clean, structured columns from the deterministic routing
and Gemini enrichment steps, so Gold does no transformation logic — it's purely
a presentation layer.

`gold_triage` renames a few technical fields into business-friendly labels
(`original_muggle_complaint`, `house_assignment`, `goblin_summary`) while keeping
every operational field — `triage_status`, `error_type`, `error_message`,
`model_version`, `retry_count`, `elapsed_seconds` — intact. Retaining that
metadata matters here: a portfolio reviewer (or a future debugging session)
can see *why* a row looks the way it does, not just the final sentiment/summary,
without having to go back to Silver.

The second query is the actual "for humans" view: it drops the operational
columns entirely and orders by house, giving a clean read of what got routed
where and how it was assessed.

In [0]:
%sql
-- GRINGOTTS GOLD LAYER
-- Silver already contains structured columns; no JSON parsing needed here.
-- Gold presents the data in a business-friendly format while retaining
-- enough operational metadata to explain the result.

CREATE OR REPLACE TABLE gringotts.gold_triage AS
SELECT
    Complaint_ID,
    Issue,
    Consumer_Complaint_Narrative AS original_muggle_complaint,
    routed_house                  AS house_assignment,
    routing_method,
    sentiment,
    gringotts_summary             AS goblin_summary,
    triage_status,
    error_type,
    error_message,
    model_version,
    retry_count,
    elapsed_seconds
FROM gringotts.silver_triage;

-- Business-ready portfolio view
SELECT
    Complaint_ID,
    Issue,
    house_assignment,
    routing_method,
    sentiment,
    goblin_summary
FROM gringotts.gold_triage
ORDER BY house_assignment, Complaint_ID;

In [0]:
%sql
-- House distribution
SELECT
    house_assignment,
    COUNT(*) AS complaint_count
FROM gringotts.gold_triage
WHERE triage_status = 'SUCCESS'
GROUP BY house_assignment
ORDER BY complaint_count DESC;

In [0]:
%sql
-- Routing methodology

SELECT
    routing_method,
    COUNT(*) AS complaint_count
FROM gringotts.gold_triage
GROUP BY routing_method
ORDER BY routing_method;

### House Responses + Dumbledore Evaluation

This step generates a written response to each successfully-triaged complaint,
then has a second, independent LLM pass grade that response — separating
*generation* from *evaluation* so the model isn't scoring its own work in the
same breath it wrote it.

**Response generation.** Each complaint is routed to its assigned House (from
Silver) and answered by that House's persona — Gryffindor advocates and pushes
accountability, Ravenclaw analyzes facts and gaps, Hufflepuff focuses on
practical next steps, Slytherin plays it strategically. Each persona prompt
explicitly forbids inventing facts, legal claims, or leverage not present in
the complaint — the goal is a distinct *approach* per house, not distinct
willingness to fabricate.

**Evaluation.** A second call — "Dumbledore" — scores that response across
seven dimensions (relevance, grounding, complaint-addressed, actionability,
professionalism, persona fidelity, completeness) on a 0–10 scale. The prompt
sets explicit anchors and tells the model 9s and 10s should be rare, since
LLM judges left unconstrained tend to cluster scores near the top of the
range regardless of actual quality.

**Scoring stays deterministic.** Dumbledore returns raw per-dimension scores
only — the weighted composite (`quality_score`) and the derived
`house_cup_points` are computed in Spark from fixed weights, not by the LLM.
This keeps the final ranking reproducible and auditable: rerunning the
weighting logic against the same scores always gives the same result.

Both API calls share one retrying helper (`call_gemini`) with the same
backoff/error-typing behavior as Silver, and every row — success or failure —
is written to `gringotts.house_responses` with full diagnostics, so a partial
or failed run is easy to debug without re-calling the API.

In [0]:
# HOUSE RESPONSES + DUMBLEDORE EVALUATION
# Goblin Triage → Routed House Agent → Dumbledore Evaluation → Deterministic House Cup Scoring
# Dumbledore judges the work. Spark keeps the score.

import json, random, time, requests
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Configuration
MODEL = "gemini-3.8-flash"
API_KEY = dbutils.secrets.get(catalog="workspace", schema="gringotts", key="gemini_api_key")
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
MAX_ATTEMPTS = 4
REQUEST_TIMEOUT = 60
MAX_NARRATIVE_CHARS = 600
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

# House Personas
HOUSE_PROMPTS = {

    "Gryffindor": """
You are a Gryffindor response agent.

Your primary objective is CONSUMER ADVOCACY and ACCOUNTABILITY.

You are bold, protective, and willing to challenge organizations when the
complaint provides evidence that something went wrong.

When appropriate:
- Clearly identify the consumer's core grievance.
- State what the organization should explain, correct, or investigate.
- Encourage the consumer to document the issue.
- Identify reasonable escalation options when ordinary resolution fails.
- Be willing to say that an explanation or correction is warranted.

Do NOT manufacture wrongdoing, legal violations, regulatory requirements,
or facts that are not present in the complaint.

Do NOT threaten the organization on the consumer's behalf.

Your response should feel like an advocate standing beside the consumer,
not a generic customer-service representative.

A strong Gryffindor response should answer:
"What should the organization be held accountable for, and what should the
consumer do if that accountability is not forthcoming?"

Be bold in tone, but disciplined in facts.
""",

    "Ravenclaw": """
You are a Ravenclaw response agent.

Your primary objective is ANALYSIS and CLARITY.

You approach the complaint like an investigator trying to determine exactly
what happened and what evidence would resolve the issue.

When appropriate:
- Break the complaint into its important factual components.
- Distinguish known facts from unanswered questions.
- Examine amounts, dates, account activity, reporting, disclosures,
  calculations, or procedures when relevant.
- Identify what documentation should be requested or reviewed.
- Explain the most useful next verification step.

Do NOT invent missing numbers, dates, policies, or account facts.

Do NOT provide generic advice when the complaint contains specific
information that can be analyzed.

Your response should feel like someone has actually examined the problem,
rather than simply categorized it.

A strong Ravenclaw response should answer:
"What do we actually know, what don't we know, and what evidence would
resolve the uncertainty?"

Precision matters more than rhetorical force.
""",

    "Hufflepuff": """
You are a Hufflepuff response agent.

Your primary objective is PRACTICAL RESOLUTION and CONSUMER SUPPORT.

You recognize that there is a person behind the complaint and focus on
helping that person make meaningful progress.

When appropriate:
- Acknowledge the specific problem described by the consumer.
- Reduce unnecessary friction.
- Recommend practical, realistic next steps.
- Explain what information or documentation the consumer should gather.
- Suggest a reasonable path toward resolution before escalation.
- Make the next step easy to understand.

Do NOT substitute empathy for substance.

Do NOT use generic phrases such as "contact customer service" unless you
explain what the consumer should ask for and why.

Do NOT invent facts, policies, legal rights, or outcomes.

Your response should feel like a knowledgeable advocate who genuinely wants
to help the consumer resolve the problem.

A strong Hufflepuff response should answer:
"What's the most practical path that could help this person resolve the
problem?"

Warmth is valuable, but usefulness comes first.
""",

    "Slytherin": """
You are a Slytherin response agent.

Your primary objective is STRATEGY and LEVERAGE.

You approach the complaint as a negotiation problem: determine what the
consumer can document, what outcome they want, and which escalation path
could improve their position.

When appropriate:
- Identify the consumer's strongest factual position.
- Recommend documentation that strengthens that position.
- Suggest a sequence of actions rather than a single generic step.
- Identify reasonable escalation channels if the initial approach fails.
- Explain what the consumer should ask the organization to do or explain.
- Consider how the consumer can preserve options before escalating.

Do NOT fabricate leverage.

Do NOT threaten lawsuits, regulators, or other consequences without factual
basis.

Do NOT claim that an organization violated a law or regulation unless that
is explicitly established in the supplied information.

Your response should feel strategic and deliberate: every recommended step
should have a purpose.

A strong Slytherin response should answer:
"How can the consumer improve their position and maximize the chance of a
useful resolution?"

Be sharp and strategic, but remain factual and professional.
"""
}

# Schemas
HOUSE_RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {"response_letter": {"type": "STRING"}},
    "required": ["response_letter"],
}

# Scores use 0-10; anchors tell Dumbledore 9-10 should be rare.
DUMBLEDORE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "relevance_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "grounding_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "complaint_addressed_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "actionability_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "professionalism_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "persona_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "completeness_score": {"type": "INTEGER", "minimum": 0, "maximum": 10},
        "evaluation_summary": {"type": "STRING"},
        "strengths": {"type": "ARRAY", "items": {"type": "STRING"}},
        "deficiencies": {"type": "ARRAY", "items": {"type": "STRING"}},
    },
    "required": [
        "relevance_score", "grounding_score", "complaint_addressed_score",
        "actionability_score", "professionalism_score", "persona_score",
        "completeness_score", "evaluation_summary", "strengths", "deficiencies",
    ],
}


def _backoff(attempt, response=None):
    retry_after = response.headers.get("Retry-After") if response is not None else None
    if retry_after:
        try:
            return float(retry_after)
        except ValueError:
            pass
    return 2 ** (attempt - 1) + random.uniform(0, 0.5)


# Generic Gemini Call (with retry + backoff)
def call_gemini(prompt, response_schema):
    headers = {"Content-Type": "application/json", "x-goog-api-key": API_KEY}
    payload = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {
            "thinkingConfig": {"thinkingLevel": "low"},
            "temperature": 0.9,
            "maxOutputTokens": 1024,
            "responseMimeType": "application/json",
            "responseSchema": response_schema,
        },
    }
    last_error_type = last_error_message = last_http_status = None

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.post(ENDPOINT, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
            last_http_status = response.status_code

            if response.status_code == 200:
                body = response.json()
                candidates = body.get("candidates", [])
                if not candidates:
                    raise ValueError("NO_CANDIDATE")
                if candidates[0].get("finishReason") == "MAX_TOKENS":
                    raise ValueError("MAX_OUTPUT_TOKENS")
                text_parts = [p.get("text", "") for p in candidates[0].get("content", {}).get("parts", []) if p.get("text")]
                if not text_parts:
                    raise ValueError("EMPTY_RESPONSE")
                parsed = json.loads("".join(text_parts))
                return {"success": True, "data": parsed, "error_type": None, "error_message": None,
                        "http_status": 200, "retry_count": attempt - 1,
                        "model_version": body.get("modelVersion", MODEL)}

            last_error_type, last_error_message = f"HTTP_{response.status_code}", response.text[:2000]
            if response.status_code not in RETRYABLE_STATUS_CODES:
                break
            if attempt < MAX_ATTEMPTS:
                time.sleep(_backoff(attempt, response))

        except requests.Timeout as exc:
            last_error_type, last_error_message = "TIMEOUT", str(exc)
            if attempt < MAX_ATTEMPTS:
                time.sleep(_backoff(attempt))
        except requests.RequestException as exc:
            last_error_type, last_error_message = "CONNECTION_ERROR", str(exc)
            if attempt < MAX_ATTEMPTS:
                time.sleep(_backoff(attempt))
        except json.JSONDecodeError as exc:
            last_error_type, last_error_message = "MODEL_JSON_PARSE_ERROR", str(exc)
            break
        except ValueError as exc:
            last_error_type, last_error_message = str(exc), str(exc)
            break
        except Exception as exc:
            last_error_type, last_error_message = "UNEXPECTED_ERROR", str(exc)
            break

    return {"success": False, "data": None, "error_type": last_error_type,
            "error_message": last_error_message, "http_status": last_http_status,
            "retry_count": MAX_ATTEMPTS - 1, "model_version": MODEL}


# Load Silver records
records = (
    spark.table("gringotts.silver_triage")
    .filter("triage_status = 'SUCCESS'")
    .select("Complaint_ID", "Issue", "Consumer_Complaint_Narrative",
            "routed_house", "routing_method", "sentiment", "gringotts_summary")
    .collect()
)

# Result schema
result_schema = StructType([
    StructField("Complaint_ID", StringType(), True),
    StructField("routed_house", StringType(), True),
    StructField("response_status", StringType(), True),
    StructField("house_response", StringType(), True),
    StructField("evaluation_summary", StringType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("retry_count", IntegerType(), True),
    StructField("model_version", StringType(), True),
    StructField("elapsed_seconds", DoubleType(), True),
    StructField("relevance_score", IntegerType(), True),
    StructField("grounding_score", IntegerType(), True),
    StructField("complaint_addressed_score", IntegerType(), True),
    StructField("actionability_score", IntegerType(), True),
    StructField("professionalism_score", IntegerType(), True),
    StructField("persona_score", IntegerType(), True),
    StructField("completeness_score", IntegerType(), True),
    StructField("strengths", StringType(), True),
    StructField("deficiencies", StringType(), True),
    StructField("quality_score", DoubleType(), True),
    StructField("house_cup_points", DoubleType(), True),
])
RESULT_FIELDS = [f.name for f in result_schema.fields]


def _row(complaint_id, house, status, house_response=None, evaluation_summary=None,
         error_type=None, error_message=None, http_status=None, retry_count=None,
         model_version=None, elapsed_seconds=None, scores=None,
         strengths=None, deficiencies=None, quality_score=None, house_cup_points=None):
    scores = scores or {}
    values = {
        "Complaint_ID": complaint_id, "routed_house": house, "response_status": status,
        "house_response": house_response, "evaluation_summary": evaluation_summary,
        "error_type": error_type, "error_message": error_message, "http_status": http_status,
        "retry_count": retry_count, "model_version": model_version, "elapsed_seconds": elapsed_seconds,
        "relevance_score": scores.get("relevance_score"),
        "grounding_score": scores.get("grounding_score"),
        "complaint_addressed_score": scores.get("complaint_addressed_score"),
        "actionability_score": scores.get("actionability_score"),
        "professionalism_score": scores.get("professionalism_score"),
        "persona_score": scores.get("persona_score"),
        "completeness_score": scores.get("completeness_score"),
        "strengths": strengths, "deficiencies": deficiencies,
        "quality_score": quality_score, "house_cup_points": house_cup_points,
    }
    return tuple(values[f] for f in RESULT_FIELDS)


# Process records
SCORE_FIELDS = ["relevance_score", "grounding_score", "complaint_addressed_score",
                "actionability_score", "professionalism_score", "persona_score", "completeness_score"]

results = []
for row in records:
    complaint_id = str(row["Complaint_ID"])
    house = row["routed_house"]
    narrative = (row["Consumer_Complaint_Narrative"] or "")[:MAX_NARRATIVE_CHARS]
    issue = row["Issue"] or ""
    sentiment = row["sentiment"] or ""
    goblin_summary = row["gringotts_summary"] or ""

    # House response
    response_prompt = f"""
{HOUSE_PROMPTS[house]}

Respond to the following consumer complaint.

CFPB Issue:
{issue}

Goblin Triage Sentiment:
{sentiment}

Goblin Summary:
{goblin_summary}

Original Consumer Complaint:
{narrative}

Write a concise response that reflects your House persona.

The response must:
- directly address the complaint
- remain grounded in the supplied information
- avoid inventing facts
- avoid unsupported legal claims
- provide practical next steps where appropriate
- remain professional

Return only the requested structured response.
"""
    response_start = time.time()
    house_result = call_gemini(response_prompt, HOUSE_RESPONSE_SCHEMA)
    response_elapsed = time.time() - response_start

    if not house_result["success"]:
        results.append(_row(complaint_id, house, "ERROR",
            error_type=house_result["error_type"], error_message=house_result["error_message"],
            http_status=house_result["http_status"], retry_count=house_result["retry_count"],
            model_version=house_result["model_version"], elapsed_seconds=response_elapsed))
        continue

    house_response = house_result["data"].get("response_letter", "").strip()
    if not house_response:
        results.append(_row(complaint_id, house, "ERROR",
            error_type="EMPTY_RESPONSE", error_message="House agent returned an empty response.",
            http_status=200, retry_count=house_result["retry_count"],
            model_version=house_result["model_version"], elapsed_seconds=response_elapsed))
        continue

    # Dumbledore evaluation
    evaluation_prompt = f"""
You are Dumbledore, the evaluator for the Hogwarts House Cup.

Your task is to evaluate the quality of ONE House response to a consumer
complaint.

Do not reward a response simply because it sounds polished.

Evaluate the actual evidence in the response.

A response should generally score in the middle of the scale unless it
demonstrates strong evidence of quality.

SCORING ANCHORS:

0-2 = absent, fundamentally wrong, or seriously deficient
3-4 = weak; important problems remain
5-6 = adequate; useful but noticeably incomplete or generic
7-8 = strong; addresses the task well with minor shortcomings
9 = exceptional; unusually thorough, specific, and well-grounded
10 = exemplary; nearly flawless and difficult to improve

A 9 or 10 should be RARE.

Do not give high scores merely because the response is grammatically
correct, polite, or generally reasonable.

------------------------------------------------------------
EVALUATION DIMENSIONS
------------------------------------------------------------

RELEVANCE
Does the response directly address the actual complaint rather than merely
discussing the general topic?

GROUNDING
Are statements supported by the supplied complaint, Issue, sentiment,
and Goblin summary?

COMPLAINT ADDRESSED
Does the response actually deal with the consumer's stated problem?

ACTIONABILITY
Does it provide concrete, realistic, complaint-specific next steps?

PROFESSIONALISM
Is the response clear, appropriate, respectful, and responsible?

HOUSE PERSONA
Does the response meaningfully reflect the assigned House's approach
without becoming a caricature?

COMPLETENESS
Does the response cover the important aspects of the complaint without
unnecessary filler?

------------------------------------------------------------
IMPORTANT
------------------------------------------------------------

Generic advice should NOT receive a high actionability score.

A polished response that fails to address the actual complaint should NOT
receive a high relevance or complaint-addressed score.

Unsupported claims should reduce grounding.

A response can be professional while still being mediocre.

Do not score based on what the House agent might have intended.
Score the response that was actually written.

------------------------------------------------------------
INPUT
------------------------------------------------------------

Assigned House:
{house}

CFPB Issue:
{issue}

Goblin Sentiment:
{sentiment}

Goblin Summary:
{goblin_summary}

Original Consumer Complaint:
{narrative}

House Response:
{house_response}

------------------------------------------------------------
OUTPUT
------------------------------------------------------------

Provide:

1. A 0-10 score for each dimension.
2. A concise evaluation summary.
3. A short list of concrete strengths.
4. A short list of concrete deficiencies.

The deficiencies should identify what prevented the response from receiving
a higher score.

A response should receive a high score only when it demonstrates
house-specific excellence, not merely competent general advice.

Do not give multiple Houses similar scores simply because their responses
are similarly polished.

Look for meaningful differences in how each House approaches the problem.

A House can score highly in one dimension and poorly in another.
Do not compensate for a weakness by raising unrelated scores.
"""
    evaluation_start = time.time()
    evaluation_result = call_gemini(evaluation_prompt, DUMBLEDORE_SCHEMA)
    evaluation_elapsed = time.time() - evaluation_start
    total_elapsed = response_elapsed + evaluation_elapsed
    combined_retries = house_result["retry_count"] + evaluation_result["retry_count"]

    if not evaluation_result["success"]:
        results.append(_row(complaint_id, house, "ERROR", house_response=house_response,
            error_type=evaluation_result["error_type"], error_message=evaluation_result["error_message"],
            http_status=evaluation_result["http_status"], retry_count=combined_retries,
            model_version=evaluation_result["model_version"], elapsed_seconds=total_elapsed))
        continue

    evaluation = evaluation_result["data"]

    # Extract and validate scores
    scores = {}
    validation_error = None
    for field in SCORE_FIELDS:
        value = evaluation.get(field)
        if not isinstance(value, int) or not 0 <= value <= 10:
            validation_error = f"Invalid {field}: {value}"
            break
        scores[field] = value

    if validation_error:
        results.append(_row(complaint_id, house, "ERROR", house_response=house_response,
            error_type="SCHEMA_ERROR", error_message=validation_error, http_status=200,
            retry_count=combined_retries, model_version=evaluation_result["model_version"],
            elapsed_seconds=total_elapsed))
        continue

    # Deterministic House Cup scoring (Spark-owned weighted score)
    quality_score = round(
        scores["relevance_score"] * 0.15
        + scores["grounding_score"] * 0.20
        + scores["complaint_addressed_score"] * 0.20
        + scores["actionability_score"] * 0.20
        + scores["professionalism_score"] * 0.10
        + scores["persona_score"] * 0.05
        + scores["completeness_score"] * 0.10, 2)
    house_cup_points = round(quality_score * 10, 2)

    results.append(_row(complaint_id, house, "SUCCESS", house_response=house_response,
        evaluation_summary=evaluation.get("evaluation_summary", ""),
        http_status=200, retry_count=combined_retries,
        model_version=evaluation_result["model_version"], elapsed_seconds=total_elapsed,
        scores=scores, strengths=json.dumps(evaluation.get("strengths", [])),
        deficiencies=json.dumps(evaluation.get("deficiencies", [])),
        quality_score=quality_score, house_cup_points=house_cup_points))

# Write results
house_responses_df = spark.createDataFrame(results, schema=result_schema)
(house_responses_df.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("gringotts.house_responses"))

# Validation
display(spark.sql("""
    SELECT Complaint_ID, routed_house, response_status, house_response,
           quality_score, house_cup_points, relevance_score, grounding_score,
           complaint_addressed_score, actionability_score, professionalism_score,
           persona_score, completeness_score, evaluation_summary, strengths, deficiencies
    FROM gringotts.house_responses
    ORDER BY quality_score DESC
"""))

In [0]:
%sql
-- GRINGOTTS HOUSE CUP — standings based on response quality, not complaint volume.
-- Each successful response contributes its normalized quality score.

CREATE OR REPLACE TABLE gringotts.gold_house_cup AS
SELECT
    routed_house AS house,
    COUNT(*) AS responses_evaluated,
    ROUND(AVG(quality_score), 2) AS average_quality_score,
    ROUND(SUM(house_cup_points), 2) AS total_house_cup_points,
    ROUND(AVG(relevance_score), 2) AS avg_relevance,
    ROUND(AVG(grounding_score), 2) AS avg_grounding,
    ROUND(AVG(complaint_addressed_score), 2) AS avg_complaint_addressed,
    ROUND(AVG(actionability_score), 2) AS avg_actionability,
    ROUND(AVG(professionalism_score), 2) AS avg_professionalism,
    ROUND(AVG(persona_score), 2) AS avg_persona,
    ROUND(AVG(completeness_score), 2) AS avg_completeness
FROM gringotts.house_responses
WHERE response_status = 'SUCCESS'
GROUP BY routed_house;

-- HOUSE CUP LEADERBOARD
SELECT house, responses_evaluated, average_quality_score, total_house_cup_points
FROM gringotts.gold_house_cup
ORDER BY average_quality_score DESC;

-- HOUSE RESPONSE QUALITY
SELECT
    house, average_quality_score, avg_relevance, avg_grounding,
    avg_complaint_addressed, avg_actionability, avg_professionalism,
    avg_persona, avg_completeness
FROM gringotts.gold_house_cup
ORDER BY average_quality_score DESC;

### Dashboard Gold Views — Presentation Layer

This step defines nine views purely for dashboard consumption — no AI
inference, aggregation logic, or business classification happens here.
Everything is a straight projection or a `GROUP BY`/`COUNT` over datasets
already finalized upstream (`gold_triage`, `house_responses`, and
`gold_house_cup`), so these views can be swapped or extended freely without
touching the pipeline that produced the underlying data.

Each view backs a specific piece of the dashboard:

- **`gold_dashboard_house_cup`** — the leaderboard: quality-weighted House
  standings, not response volume.
- **`gold_dashboard_response_quality`** — per-complaint scores across all
  seven evaluation dimensions, for comparing Houses side by side.
- **`gold_dashboard_complaints`** / **`gold_dashboard_complaint_summary`** —
  complaint-level detail and a pre-aggregated sentiment/volume rollup, so the
  dashboard doesn't need to aggregate on the fly.
- **`gold_dashboard_issue_distribution`** — complaint volume by CFPB Issue
  and House, for spotting concentration patterns.
- **`gold_dashboard_operations`** — pipeline health: error counts, retry
  rates, and latency by triage status, separate from the business-facing
  views above it.
- **`gold_dashboard_routing`** — how much of the routing came from the
  deterministic Issue mapping versus the load-balancing fallback.
- **`gold_dashboard_explorer`** — the full lineage view, joining triage
  through to response and evaluation for drilling into any single complaint.
- **`gold_dashboard_house_dimensions`** — the House Cup rollup unpivoted into
  long form (one row per house/dimension), shaped for a radar or grouped
  bar chart rather than a wide table.

The four `SELECT` statements at the end are validation only — confirming the
views return sane data before wiring them into the actual dashboard.

In [0]:
%sql
SELECT house, total_house_cup_points, average_quality_score 
FROM gringotts.gold_dashboard_house_cup 
ORDER BY total_house_cup_points DESC;

In [0]:
%sql
SELECT house, dimension, score 
FROM gringotts.gold_dashboard_house_dimensions;

In [0]:
%sql
SELECT
    Complaint_ID, Issue, house_assignment, routing_method, sentiment,
    triage_status, model_version, retry_count, elapsed_seconds
FROM gringotts.gold_triage;


In [0]:
%sql
SELECT house_assignment AS house, Issue, COUNT(*) AS complaint_count
FROM gringotts.gold_triage
WHERE triage_status = 'SUCCESS'
GROUP BY house_assignment, Issue;

In [0]:
%sql
-- GRINGOTTS DASHBOARD GOLD VIEWS
-- Presentation-oriented projections over canonical Gold datasets.
-- No AI inference or business classification occurs here.

-- 1. HOUSE CUP — Primary dataset for the House Cup leaderboard.
-- Quality, not response volume, determines the standings.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_house_cup AS
SELECT
    house, responses_evaluated, average_quality_score, total_house_cup_points,
    avg_relevance, avg_grounding, avg_complaint_addressed, avg_actionability,
    avg_professionalism, avg_persona, avg_completeness
FROM gringotts.gold_house_cup;

-- 2. RESPONSE QUALITY — Long-form evaluation dataset for comparing quality dimensions across Houses.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_response_quality AS
SELECT
    routed_house AS house, Complaint_ID, quality_score,
    relevance_score, grounding_score, complaint_addressed_score,
    actionability_score, professionalism_score, persona_score, completeness_score,
    house_cup_points, evaluation_summary
FROM gringotts.house_responses
WHERE response_status = 'SUCCESS';

-- 3. COMPLAINT INTELLIGENCE — Complaint-level analytical dataset for volume, sentiment, issue, and routing.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_complaints AS
SELECT
    Complaint_ID, Issue, house_assignment, routing_method, sentiment,
    triage_status, model_version, retry_count, elapsed_seconds
FROM gringotts.gold_triage;

-- 4. COMPLAINT SUMMARY — Pre-aggregated dataset for complaint volume and sentiment visualizations.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_complaint_summary AS
SELECT house_assignment AS house, sentiment, COUNT(*) AS complaint_count
FROM gringotts.gold_triage
WHERE triage_status = 'SUCCESS'
GROUP BY house_assignment, sentiment;

-- 5. ISSUE DISTRIBUTION — Complaint volume by CFPB Issue and House.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_issue_distribution AS
SELECT house_assignment AS house, Issue, COUNT(*) AS complaint_count
FROM gringotts.gold_triage
WHERE triage_status = 'SUCCESS'
GROUP BY house_assignment, Issue;

-- 6. PIPELINE OPERATIONS — Operational metrics: AI processing success, retries, and latency.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_operations AS
SELECT
    triage_status,
    COUNT(*) AS record_count,
    SUM(CASE WHEN error_type IS NOT NULL THEN 1 ELSE 0 END) AS error_count,
    SUM(CASE WHEN retry_count > 0 THEN 1 ELSE 0 END) AS records_retried,
    ROUND(AVG(elapsed_seconds), 2) AS avg_elapsed_seconds,
    ROUND(MAX(elapsed_seconds), 2) AS max_elapsed_seconds
FROM gringotts.gold_triage
GROUP BY triage_status;

-- 7. ROUTING ANALYSIS — Issue mapping vs deterministic balancing fallback.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_routing AS
SELECT routing_method, house_assignment AS house, COUNT(*) AS complaint_count
FROM gringotts.gold_triage
GROUP BY routing_method, house_assignment;

-- 8. COMPLAINT EXPLORER — Full complaint-level lineage: complaint → triage → response → evaluation.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_explorer AS
SELECT
    g.Complaint_ID, g.Issue, g.original_muggle_complaint, g.house_assignment,
    g.routing_method, g.sentiment, g.goblin_summary,
    r.house_response, r.quality_score, r.house_cup_points,
    r.relevance_score, r.grounding_score, r.complaint_addressed_score,
    r.actionability_score, r.professionalism_score, r.persona_score,
    r.completeness_score, r.evaluation_summary, r.response_status,
    r.retry_count AS response_retry_count, r.elapsed_seconds AS response_elapsed_seconds,
    g.model_version AS triage_model_version, r.model_version AS response_model_version
FROM gringotts.gold_triage g
LEFT JOIN gringotts.house_responses r ON g.Complaint_ID = r.Complaint_ID;

-- 9. HOUSE CUP QUALITY DIMENSIONS — Compact dataset for radar-style or grouped comparison.
CREATE OR REPLACE VIEW gringotts.gold_dashboard_house_dimensions AS
SELECT house, 'Relevance' AS dimension, avg_relevance AS score FROM gringotts.gold_house_cup
UNION ALL
SELECT house, 'Grounding', avg_grounding FROM gringotts.gold_house_cup
UNION ALL
SELECT house, 'Complaint Addressed', avg_complaint_addressed FROM gringotts.gold_house_cup
UNION ALL
SELECT house, 'Actionability', avg_actionability FROM gringotts.gold_house_cup
UNION ALL
SELECT house, 'Professionalism', avg_professionalism FROM gringotts.gold_house_cup
UNION ALL
SELECT house, 'Persona', avg_persona FROM gringotts.gold_house_cup
UNION ALL
SELECT house, 'Completeness', avg_completeness FROM gringotts.gold_house_cup;

-- QUICK VALIDATION
SELECT * FROM gringotts.gold_dashboard_house_cup ORDER BY average_quality_score DESC;
SELECT * FROM gringotts.gold_dashboard_operations;
SELECT * FROM gringotts.gold_dashboard_routing;
SELECT * FROM gringotts.gold_dashboard_explorer LIMIT 10;